# План выполнения домашнего задания

## Часть 1. Парсинг

1. **Определить источник данных и целевую переменную**
   - Выбрать сайт без API.
   - Зафиксировать, что будет текстом (`text`) и что будет целевой переменной (`target`).
   - Определить тип таргета: непрерывный (для регрессии) или дискретный (для классификации).

2. **Реализовать парсер**
   - Собрать ссылки на страницы объектов.
   - Для каждой страницы извлечь нужные поля (`title`, `text`, `target`, при необходимости `date`, `url`).
   - Добавить задержку между запросами: `time.sleep(0.3)`.
   - Обработать ошибки (`try/except`) и невалидные страницы.

3. **Сформировать датасет**
   - Собрать `pandas.DataFrame` с минимумом колонок: `text`, `target`.
   - Очистить данные: удалить дубликаты и пустые значения.
   - Сохранить результат в `csv` для воспроизводимости.

## Часть 2. NLP

4. **Краткий EDA (разведочный анализ)**
   - Посмотреть размер датасета, пропуски, длины текстов.
   - Проверить распределение целевой переменной (и баланс классов, если это классификация).

5. **Разделение на train/test**
   - Выполнить `train_test_split` с тестовой долей 20-30%.
   - Для классификации использовать `stratify=y`.

6. **TF-IDF преобразование текста**
   - Настроить `TfidfVectorizer`:
     - `ngram_range=(1, 2)` (слова и биграммы),
     - стоп-слова,
     - фильтрация редких/частых слов через `min_df` и `max_df`,
     - `norm=None` (убрать L2-нормализацию).
   - Обучить векторизатор на train и применить к train/test.

7. **Построить базовую модель**
   - Непрерывный таргет: линейная регрессия / Ridge.
   - Дискретный таргет: логистическая регрессия.

8. **Подобрать регуляризацию**
   - Использовать `GridSearchCV` или ручной перебор параметров:
     - `alpha` для Ridge,
     - `C` для LogisticRegression.
   - Выбрать лучшие параметры по кросс-валидации.

9. **Оценить качество на тесте**
   - Регрессия: `MAE`, `RMSE`, `R2`.
   - Классификация: `accuracy`, `F1`, `ROC-AUC`.

10. **Интерпретация модели**
    - Получить коэффициенты модели и признаки (`feature_names`).
    - Визуализировать топ-50 наиболее значимых слов/биграмм.
    - Кратко интерпретировать результаты.

11. **Финальное оформление ноутбука**
    - Добавить структурированные markdown-блоки и итоговые выводы.

In [7]:
import json
import random
import re
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# ------------------------------
# Настройки парсинга
# ------------------------------
CATEGORY_URL = "https://irecommend.ru/catalog/list/125438-125528"
N_CATEGORY_PAGES = 5          # сколько страниц категории просматривать
MAX_PRODUCT_PAGES = 5         # сколько страниц отзывов на 1 товар
MAX_REVIEWS_TOTAL = 10      # общий лимит отзывов
SLEEP_MIN = 0.35
SLEEP_MAX = 0.9
OUT_CSV = "drill_reviews_irecommend.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.8",
}


def pause():
    time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))


def get_soup(session, url, retries=3, timeout=20):
    for attempt in range(1, retries + 1):
        try:
            response = session.get(url, timeout=timeout)
            response.raise_for_status()
            return BeautifulSoup(response.text, "html.parser")
        except requests.HTTPError as exc:
            status = exc.response.status_code if exc.response is not None else None
            if attempt == retries:
                if status == 521:
                    print(
                        f"[WARN] Сервер iRecommend временно недоступен (HTTP 521): {url}. "
                        "Это не ошибка кода. Повторите позже или смените сеть/VPN."
                    )
                else:
                    print(f"[WARN] HTTP ошибка при открытии {url}: {exc}")
                return None
        except requests.RequestException as exc:
            if attempt == retries:
                print(f"[WARN] Сетевая ошибка при открытии {url}: {exc}")
                return None

        # Экспоненциальный backoff с небольшим джиттером
        time.sleep((1.3 ** attempt) + random.uniform(0.2, 0.8))

    return None


def unique_keep_order(items):
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result


def parse_json_ld(soup):
    data = []
    for node in soup.select('script[type="application/ld+json"]'):
        raw = (node.get_text() or "").strip()
        if not raw:
            continue
        try:
            parsed = json.loads(raw)
            if isinstance(parsed, list):
                data.extend(parsed)
            else:
                data.append(parsed)
        except Exception:
            continue
    return data


def extract_product_links_from_category(soup):
    links = []

    # На iRecommend страницы товаров обычно содержат /catalog/reviews/
    for a in soup.select('a[href*="/catalog/reviews/"]'):
        href = a.get("href")
        if href:
            links.append(urljoin("https://irecommend.ru", href))

    # Fallback: если структура страницы изменилась
    if not links:
        for a in soup.select("a[href]"):
            href = a.get("href")
            if not href:
                continue
            full = urljoin("https://irecommend.ru", href)
            if "/catalog/reviews/" in full:
                links.append(full)

    return unique_keep_order(links)


def extract_review_links_from_product_page(soup):
    links = []

    # Ссылки на отдельные отзывы часто имеют /content/
    for a in soup.select('a[href*="/content/"]'):
        href = a.get("href")
        if not href:
            continue
        full = urljoin("https://irecommend.ru", href)
        if "/content/" in full and "irecommend.ru" in full:
            links.append(full)

    return unique_keep_order(links)


def extract_product_page_urls(product_url, first_soup, max_pages=MAX_PRODUCT_PAGES):
    page_urls = [product_url]

    # Пробуем найти прямые ссылки пагинации
    discovered = []
    for a in first_soup.select("a[href]"):
        href = a.get("href") or ""
        if "page=" in href and "/catalog/reviews/" in href:
            discovered.append(urljoin("https://irecommend.ru", href))

    discovered = unique_keep_order(discovered)

    if discovered:
        for u in discovered:
            if u not in page_urls:
                page_urls.append(u)
            if len(page_urls) >= max_pages:
                break
    else:
        # Fallback: наиболее частый шаблон пагинации
        for page in range(2, max_pages + 1):
            page_urls.append(f"{product_url}?page={page}")

    return page_urls[:max_pages]


def clean_text(text):
    if text is None:
        return ""
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_review_data(session, review_url):
    soup = get_soup(session, review_url)
    pause()
    if soup is None:
        return None

    title = ""
    h1 = soup.select_one("h1")
    if h1:
        title = clean_text(h1.get_text(" ", strip=True))

    review_text = ""
    text_selectors = [
        '[itemprop="reviewBody"]',
        ".reviewText",
        ".description",
        ".field-name-body",
        ".content",
    ]
    for sel in text_selectors:
        node = soup.select_one(sel)
        if node:
            candidate = clean_text(node.get_text(" ", strip=True))
            if len(candidate) > len(review_text):
                review_text = candidate

    # Fallback: взять самый длинный абзац
    if len(review_text) < 80:
        paragraphs = [clean_text(p.get_text(" ", strip=True)) for p in soup.select("p")]
        paragraphs = [p for p in paragraphs if len(p) > 50]
        if paragraphs:
            review_text = max(paragraphs, key=len)

    rating = None
    rating_node = soup.select_one('meta[itemprop="ratingValue"]')
    if rating_node and rating_node.get("content"):
        try:
            rating = float(rating_node.get("content").replace(",", "."))
        except Exception:
            rating = None

    date_published = ""
    date_node = soup.select_one('meta[itemprop="datePublished"]')
    if date_node and date_node.get("content"):
        date_published = date_node.get("content")
    else:
        time_node = soup.select_one("time")
        if time_node:
            date_published = clean_text(time_node.get_text(" ", strip=True))

    product_name = ""

    # JSON-LD часто содержит rating/itemReviewed
    for obj in parse_json_ld(soup):
        if isinstance(obj, dict):
            if rating is None and "ratingValue" in obj:
                try:
                    rating = float(str(obj["ratingValue"]).replace(",", "."))
                except Exception:
                    pass
            item_reviewed = obj.get("itemReviewed")
            if not product_name and isinstance(item_reviewed, dict):
                product_name = clean_text(str(item_reviewed.get("name", "")))

    if not product_name:
        breadcrumb = soup.select('a[href*="/catalog/reviews/"]')
        if breadcrumb:
            product_name = clean_text(breadcrumb[-1].get_text(" ", strip=True))

    # Базовая валидация
    if not review_text or rating is None:
        return None

    return {
        "product_name": product_name,
        "review_title": title,
        "review_text": review_text,
        "rating": rating,
        "review_date": date_published,
        "review_url": review_url,
    }


# ------------------------------
# 1) Сбор ссылок на товары из категории
# ------------------------------
session = requests.Session()
session.headers.update(HEADERS)

category_product_links = []
for page in tqdm(range(1, N_CATEGORY_PAGES + 1), desc="Category pages"):
    page_url = CATEGORY_URL if page == 1 else f"{CATEGORY_URL}?page={page}"
    soup = get_soup(session, page_url)
    pause()
    if soup is None:
        continue

    links = extract_product_links_from_category(soup)
    category_product_links.extend(links)

category_product_links = unique_keep_order(category_product_links)
print(f"Найдено товаров: {len(category_product_links)}")


# ------------------------------
# 2) Сбор ссылок на отзывы со страниц товаров
# ------------------------------
all_review_links = []

for product_url in tqdm(category_product_links, desc="Product pages"):
    soup_first = get_soup(session, product_url)
    pause()
    if soup_first is None:
        continue

    product_pages = extract_product_page_urls(product_url, soup_first, max_pages=MAX_PRODUCT_PAGES)

    for p_url in product_pages:
        soup_p = soup_first if p_url == product_url else get_soup(session, p_url)
        pause()
        if soup_p is None:
            continue

        review_links = extract_review_links_from_product_page(soup_p)
        all_review_links.extend(review_links)

        if len(all_review_links) >= MAX_REVIEWS_TOTAL * 2:
            # С запасом: часть ссылок может оказаться дубликатами/невалидными
            break

    if len(all_review_links) >= MAX_REVIEWS_TOTAL * 2:
        break

all_review_links = unique_keep_order(all_review_links)
print(f"Найдено ссылок на отзывы: {len(all_review_links)}")


# ------------------------------
# 3) Парсинг самих отзывов
# ------------------------------
rows = []
for review_url in tqdm(all_review_links, desc="Review pages"):
    data = extract_review_data(session, review_url)
    if data is not None:
        rows.append(data)

    if len(rows) >= MAX_REVIEWS_TOTAL:
        break

print(f"Успешно распарсено отзывов: {len(rows)}")


# ------------------------------
# 4) Очистка и сохранение датасета
# ------------------------------
df = pd.DataFrame(rows)

if not df.empty:
    before = len(df)
    df = df.drop_duplicates(subset=["review_url"]).copy()
    df["review_text"] = df["review_text"].astype(str).map(clean_text)
    df = df[df["review_text"].str.len() > 30].copy()
    df = df[df["rating"].notna()].copy()

    print(f"Удалено строк при очистке: {before - len(df)}")

    # Колонки под NLP-часть ДЗ
    df["text"] = df["review_text"]
    df["target"] = df["rating"]

    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Сохранено в: {OUT_CSV}")

    print("\nРазмер датасета:", df.shape)
    print("\nРаспределение rating:")
    print(df["rating"].value_counts(dropna=False).sort_index().to_string())

    display(df.head(5))
else:
    print("Данные не собраны. Проверь селекторы и/или увеличь лимиты страниц.")

Category pages:   0%|          | 0/5 [00:00<?, ?it/s]

[WARN] Сервер iRecommend временно недоступен (HTTP 521): https://irecommend.ru/catalog/list/125438-125528. Это не ошибка кода. Повторите позже или смените сеть/VPN.


Category pages:  20%|██        | 1/5 [00:04<00:19,  4.79s/it]

[WARN] Сервер iRecommend временно недоступен (HTTP 521): https://irecommend.ru/catalog/list/125438-125528?page=2. Это не ошибка кода. Повторите позже или смените сеть/VPN.


Category pages:  40%|████      | 2/5 [00:09<00:13,  4.62s/it]

...

Найдено товаров: 0


Product pages: 0it [00:00, ?it/s]


Найдено ссылок на отзывы: 0


Review pages: 0it [00:00, ?it/s]

Успешно распарсено отзывов: 0
Данные не собраны. Проверь селекторы и/или увеличь лимиты страниц.
